# Weather-Weather Lang: What Can Data Tell Us?
### Hands-On Notebook — Predicting Rain Tomorrow with Machine Learning

Dataset: [Australia Weather Data (Kaggle)](https://www.kaggle.com/datasets/arunavakrchakraborty/australia-weather-data)

> Theory and concept explanations are covered in the slide deck. This notebook focuses on the hands-on workflow: **Data → Features → Train → Evaluate → Improve**.

## 0. Setup

In [ ]:
# Install packages if needed (uncomment if running in a fresh environment)
# !pip install pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
%matplotlib inline

## 1. Load & Explore the Data

Download the dataset from Kaggle and place `weatherAUS.csv` in the same folder as this notebook.

In [ ]:
df = pd.read_csv("weatherAUS.csv")

In [ ]:
# Inspect the data

In [ ]:
# Show the number of rows and columns

In [ ]:
# Show the data type and non-null counts

In [ ]:
# See distribution for RainTomorrow 
# Keep an eye on this, we'll revisit it in the Evaluation section

In [ ]:
# Create a bar chart showing the RainTomorrow


In [ ]:
# Show the missing values per column

## 2. Data Preparation & Feature Selection

In [ ]:
cols_to_drop = ['row ID', 'Date', 'Location', 'Evaporation', 'Sunshine', 'Cloud9am', 'Cloud3pm']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

df.shape

In [ ]:
# RainToday is a feature -- drop rows where it's missing, then encode Yes/No as 1/0
df = df.dropna(subset=['RainToday'])
df['RainToday'] = df['RainToday'].map({'Yes': 1, 'No': 0})

df['RainToday'].value_counts()

In [ ]:
# RainTomorrow is our label. Handle both encodings: text ('Yes'/'No') or already numeric (0/1)
if df['RainTomorrow'].dtype == object:
    df['RainTomorrow'] = df['RainTomorrow'].map({'Yes': 1, 'No': 0})

df = df.dropna(subset=['RainTomorrow'])
df['RainTomorrow'].value_counts()

In [ ]:
# Impute remaining missing numeric values with the column mean
num_cols = df.select_dtypes(include='number').columns
df[num_cols] = df[num_cols].fillna(df[num_cols].mean())

df.isnull().sum().sum()  # should be 0 for numeric columns now

In [ ]:
# Select a manageable, meaningful feature set
features = ['MinTemp', 'MaxTemp', 'Rainfall', 'Humidity9am', 'Humidity3pm',
            'Pressure9am', 'Pressure3pm', 'WindGustSpeed', 'RainToday']

X = df[features]
y = df['RainTomorrow']

X.head()

In [ ]:
# Quick correlation check against the label (sanity check on feature choice)
corr = df[features + ['RainTomorrow']].corr()['RainTomorrow'].sort_values(ascending=False)
corr

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df[features + ['RainTomorrow']].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title("Feature Correlation Heatmap")
plt.show()

## 3. Train/Test Split & Model Training

We'll train three models: **Logistic Regression**, **Decision Tree**, and **Random Forest**.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape, " Test size:", X_test.shape)

### 3.1 Logistic Regression

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train)

log_preds = log_model.predict(X_test_scaled)
print("Logistic Regression trained.")

### 3.2 Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_model.fit(X_train, y_train)

tree_preds = tree_model.predict(X_test)
print("Decision Tree trained.")

In [ ]:
plt.figure(figsize=(18, 8))
plot_tree(tree_model, feature_names=features, class_names=['No Rain', 'Rain'],
          filled=True, fontsize=7)
plt.show()

### 3.3 Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)
print("Random Forest trained.")

In [ ]:
# Feature importance from Random Forest -- nice bridge into "which features matter most"
importances = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=False)

plt.figure(figsize=(8,5))
importances.plot(kind='barh', color='#55A868')
plt.title("Random Forest Feature Importance")
plt.gca().invert_yaxis()
plt.show()

## 4. Evaluation

Comparing Accuracy, Precision, Recall, and F1-score across all three models.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

results = {
    'Logistic Regression': (y_test, log_preds),
    'Decision Tree': (y_test, tree_preds),
    'Random Forest': (y_test, rf_preds),
}

for name, (y_true, preds) in results.items():
    print(f"--- {name} ---")
    print("Accuracy:", round(accuracy_score(y_true, preds), 4))
    print(classification_report(y_true, preds, target_names=['No Rain', 'Rain']))
    print()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, (y_true, preds)) in zip(axes, results.items()):
    ConfusionMatrixDisplay.from_predictions(y_true, preds, ax=ax, colorbar=False)
    ax.set_title(name)

plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side metrics summary table
from sklearn.metrics import precision_score, recall_score, f1_score

summary = []
for name, (y_true, preds) in results.items():
    summary.append({
        'Model': name,
        'Accuracy': accuracy_score(y_true, preds),
        'Precision': precision_score(y_true, preds),
        'Recall': recall_score(y_true, preds),
        'F1-score': f1_score(y_true, preds),
    })

summary_df = pd.DataFrame(summary).set_index('Model').round(4)
summary_df

## 5. Overfitting & Model Improvement

Comparing training accuracy vs. test accuracy to check for overfitting.

In [ ]:
models_for_check = {
    'Logistic Regression': (log_model, X_train_scaled, X_test_scaled),
    'Decision Tree': (tree_model, X_train, X_test),
    'Random Forest': (rf_model, X_train, X_test),
}

overfit_summary = []
for name, (model, xtr, xte) in models_for_check.items():
    train_acc = accuracy_score(y_train, model.predict(xtr))
    test_acc = accuracy_score(y_test, model.predict(xte))
    overfit_summary.append({
        'Model': name,
        'Train Accuracy': train_acc,
        'Test Accuracy': test_acc,
        'Gap': train_acc - test_acc,
    })

overfit_df = pd.DataFrame(overfit_summary).set_index('Model').round(4)
overfit_df

In [ ]:
# Demonstration: unlimited-depth tree (prone to overfitting) vs. limited-depth tree
overfit_tree = DecisionTreeClassifier(random_state=42)  # no max_depth limit
overfit_tree.fit(X_train, y_train)

train_acc_overfit = accuracy_score(y_train, overfit_tree.predict(X_train))
test_acc_overfit = accuracy_score(y_test, overfit_tree.predict(X_test))

print(f"Unlimited-depth tree  -> Train: {train_acc_overfit:.4f}  Test: {test_acc_overfit:.4f}  Gap: {train_acc_overfit - test_acc_overfit:.4f}")
print(f"max_depth=4 tree      -> Train: {overfit_df.loc['Decision Tree', 'Train Accuracy']:.4f}  Test: {overfit_df.loc['Decision Tree', 'Test Accuracy']:.4f}  Gap: {overfit_df.loc['Decision Tree', 'Gap']:.4f}")

In [ ]:
depths = range(1, 21)
train_scores, test_scores = [], []

for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, m.predict(X_train)))
    test_scores.append(accuracy_score(y_test, m.predict(X_test)))

plt.figure(figsize=(9,6))
plt.plot(depths, train_scores, marker='o', label='Train Accuracy')
plt.plot(depths, test_scores, marker='o', label='Test Accuracy')
plt.xlabel("Tree Depth (max_depth)")
plt.ylabel("Accuracy")
plt.title("Overfitting Curve: Train vs. Test Accuracy by Tree Depth")
plt.legend()
plt.show()

## 6. Try It Yourself

A few quick experiments to run live or leave as a takeaway:

1. Swap in a different feature set (e.g., add `WindGustDir` with one-hot encoding) — does performance change?
2. Try `RandomForestClassifier(n_estimators=200)` — does more trees help?
3. Try `class_weight='balanced'` in Logistic Regression or Random Forest — does it change precision/recall trade-offs?
4. Use `GridSearchCV` to tune `max_depth` automatically instead of guessing.

---
### Resources
- Dataset: https://www.kaggle.com/datasets/arunavakrchakraborty/australia-weather-data
- scikit-learn docs: https://scikit-learn.org/stable/documentation.html

*End of notebook — thank you for joining Weather-Weather Lang!* 🌦️